In [16]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [17]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [18]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)


In [19]:
dataset = generate_dataset()


with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [20]:
def run_prompt(test_case):
    #combines boths the prompts and the test cases and provide these as messages to claude
    prompt = f"""
    Please solve the following task:

    {test_case["task"]}
    """
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output



In [21]:
def run_test_case(test_case):
    #returns json with the test case, and output from claude with the score
    output = run_prompt(test_case)

    #todo -grading
    score = 10
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [22]:
def run_eval(test_case):
    results =  []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    return results

In [23]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)
results = run_eval(dataset)

In [24]:
print(json.dumps(results, indent = 2))

[
  {
    "output": "# AWS Region Extraction from S3 URL\n\nHere's a Python function that extracts the AWS region from an S3 bucket URL:\n\n```python\nimport re\nfrom typing import Optional\n\ndef extract_aws_region_from_s3_url(url: str) -> Optional[str]:\n    \"\"\"\n    Extracts the AWS region from an S3 bucket URL.\n    \n    Args:\n        url: S3 URL in formats like:\n            - s3://my-bucket.s3.us-west-2.amazonaws.com/key\n            - https://my-bucket.s3.us-west-2.amazonaws.com/key\n            - s3://my-bucket.s3.amazonaws.com/key (returns None, us-east-1)\n            - s3://my-bucket/key (returns None)\n    \n    Returns:\n        AWS region string (e.g., 'us-west-2') or None if region not found\n    \"\"\"\n    # Pattern to match region in S3 URL\n    # Matches: .s3.<region>.amazonaws.com or .s3-<region>.amazonaws.com\n    pattern = r'\\.s3[.-]([a-z0-9\\-]+)\\.amazonaws\\.com'\n    \n    match = re.search(pattern, url)\n    if match:\n        return match.group(1)\n   